# Cortex Agent Streaming Demo

This notebook demonstrates how to call the **Snowflake Cortex Agent REST API**
directly with **SSE streaming**, parse the event stream in pure Python, and display
rich results (text, SQL, tables, charts).

1. **Setup** — imports, session, JWT
2. **Rich display** — `display_result()` with thinking, badges, SQL, DataFrames, charts
3. **One-shot** — `run_agent()` for single questions
4. **Multi-turn** — `AgentChat` with local history
5. **Thread mode** — server-side continuity via `create_thread()`
6. **Quiet collection** — `collect_agent_events()` for pipelines
7. **Custom streaming** — `iter_normalized_agent_events()` for custom UIs
8. **Raw event inspector** — see every SSE event from the wire

## 1. Setup and Configuration

In [1]:
#| eval: false
import json
import pandas as pd
from IPython.display import display, Markdown, HTML

from mcp_ski_resort.core import (
    default_session, get_headers,
    stream_agent_sse, normalize_event, AgentResult,
    run_agent, collect_agent_events, iter_normalized_agent_events,
    result_set_to_dataframe,
    AgentChat, create_thread,
)

session = default_session()
token = session.jwt_gen.get_token()
print(f"Account: {session.account}")
print(f"Host:    {session.host}")
print(f"JWT:     {token[:40]}...OK")

Account: trb65519
Host:    https://trb65519.snowflakecomputing.com
JWT:     eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCJ9.eyJ...OK


## 2. Rich Display Helper

Renders the full `AgentResult` with collapsible thinking, tool-use badges,
Markdown answer, syntax-highlighted SQL, DataFrames, Altair charts, errors,
and a status bar.

In [2]:
#| eval: false
def display_result(result: AgentResult):

    if result.thinking:
        thinking_text = "\n\n".join(result.thinking)
        display(HTML(
            f'<details style="margin-bottom:12px;">'
            f'<summary style="cursor:pointer;font-weight:600;color:#6b7280;">'
            f'Thinking ({len(result.thinking)} steps)</summary>'
            f'<div style="padding:8px 16px;background:#f9fafb;border-radius:8px;margin-top:4px;'
            f'font-size:13px;color:#4b5563;white-space:pre-wrap;">{thinking_text[:2000]}</div>'
            f'</details>'
        ))

    if result.tools_used:
        badges = " ".join(
            f'<span style="display:inline-block;padding:2px 8px;margin:2px;background:#eff6ff;'
            f'color:#2563eb;border-radius:9999px;font-size:12px;border:1px solid #bfdbfe;">'
            f'{t}</span>'
            for t in result.tools_used
        )
        display(HTML(f'<div style="margin-bottom:8px;">{badges}</div>'))

    if result.answer:
        display(Markdown(result.answer))

    for i, sql in enumerate(result.sql_queries):
        display(HTML(
            f'<details style="margin:8px 0;">'
            f'<summary style="cursor:pointer;font-weight:600;color:#6b7280;">'
            f'SQL Query {i+1}</summary>'
            f'<pre style="padding:12px;background:#1e293b;color:#e2e8f0;border-radius:8px;'
            f'overflow-x:auto;font-size:13px;">{sql}</pre>'
            f'</details>'
        ))

    for i, df in enumerate(result.dataframes):
        display(HTML(f'<p style="font-weight:600;color:#374151;margin:8px 0 4px;">Result {i+1} ({len(df)} rows)</p>'))
        display(df.head(20))

    for i, spec in enumerate(result.chart_specs):
        try:
            import altair as alt
            chart = alt.Chart.from_dict(spec)
            display(chart)
        except ImportError:
            display(HTML(
                f'<details><summary>Chart Spec {i+1} (install altair to render)</summary>'
                f'<pre>{json.dumps(spec, indent=2)[:2000]}</pre></details>'
            ))
        except Exception:
            display(HTML(
                f'<details><summary>Chart Spec {i+1} (raw JSON)</summary>'
                f'<pre>{json.dumps(spec, indent=2)[:2000]}</pre></details>'
            ))

    if result.errors:
        for err in result.errors:
            display(HTML(
                f'<div style="padding:12px;background:#fef2f2;border:1px solid #fecaca;'
                f'border-radius:8px;color:#991b1b;">{err}</div>'
            ))

    display(HTML(
        f'<div style="margin-top:12px;padding:8px 12px;background:#f0fdf4;border-radius:8px;'
        f'font-size:12px;color:#166534;">'
        f'Completed in {result.duration_seconds}s | '
        f'Tools: {len(result.tools_used)} | '
        f'SQL queries: {len(result.sql_queries)} | '
        f'Result sets: {len(result.dataframes)} | '
        f'Charts: {len(result.chart_specs)}'
        f'</div>'
    ))

## 3. Demo: Resort Executive Agent

The `RESORT_EXECUTIVE` agent has access to all 11 semantic views and can
synthesize across domains.

In [3]:
#| eval: false
result = run_agent(
    agent_name="RESORT_EXECUTIVE",
    question="Give me a complete resort performance summary for the 2024-2025 season",
)
display_result(result)

Calling RESORT_EXECUTIVE with: 'Give me a complete resort performance summary for the 2024-2025 season'
------------------------------------------------------------
  [status] Planning the next steps
  [status] Choosing data sources to use
  [status] Getting additional context
  [tool] DailySummaryKPIs
  [status] Streaming SQL from DailySummaryKPIs
  [sql] WITH __fact_pass_usage AS (
  SELECT
    customer_key,
    date_key,
    usage_key,
    hours_on_mou...
  [table] 1 rows x 19 cols
  [status] Reviewing the results
  [status] Rethinking the plan
  [status] Planning the next steps
  [status] Choosing data sources to use
  [status] Getting additional context
  [tool] LiftOperations
  [status] Streaming SQL from LiftOperations
  [tool] RevenueAnalytics
  [status] Streaming SQL from RevenueAnalytics
  [sql] WITH lift_ops AS (
    SELECT *
    FROM SEMANTIC_VIEW(
        AM_SKI_RESORT.SEMANTIC.SEM_OPERATION...
  [table] 0 rows x 0 cols
  [sql] WITH ticket_rev AS (
    SELECT *
    FROM SE


# 2024-2025 Season Performance Summary

## Season Overview
**Period:** November 1, 2024 - April 30, 2025 (Complete season)

The 2024-2025 season delivered strong performance across visitation, revenue, and guest engagement metrics, with pass holders driving the majority of activity and excellent snow conditions supporting nearly half of all visits.

---

## Key Performance Highlights

### Visitation & Guest Engagement


**Guest Activity:**
- **103,597 total visits** from **7,803 unique visitors**
- **13.28 visits per guest** - indicating strong season pass utilization and repeat visitation
- **1.62 million lift rides** with an average of **15.68 rides per visit**
- Guests spent an average of **6.65 hours on mountain** per visit
- **76.8% pass holder composition** (79,556 visits) vs. 23.2% day ticket guests (24,041 visits)

**Visit Patterns:**
- **60.6% weekday visits** (62,789) vs. **39.4% weekend** (40,808) - showing strong midweek engagement
- **36,729 holiday visits** captured peak demand periods
- **44.2% of visits occurred on excellent snow days** (45,828 visits) - demonstrating favorable conditions

---

### Revenue Performance


**Total Revenue: $7,227,690**

**Revenue Mix:**
- **Food & Beverage:** $3,094,287 (42.8%) - largest revenue stream with 216,944 transactions
- **Ticket Sales:** $2,619,109 (36.2%) - 24,041 tickets sold
- **Equipment Rentals:** $1,514,294 (20.9%) - 27,770 rental transactions

**Per-Visit Economics:**
- Average ticket price: **$108.92** per ticket
- Average rental value: **$54.52** per transaction
- Average F&B spend: **$14.26** per transaction
- **Total revenue per visit: $69.78**

---

### Operational Performance

**Lift Operations:**
Lift-specific performance data (wait times, capacity utilization, maintenance metrics) is not available for the 2024-2025 season in the current dataset.

**Key Operational Indicators:**
- High lift ride efficiency with 15.68 rides per visit
- Extended guest engagement with 6.65 hours per visit on average
- Strong utilization across all day types (weekend, weekday, holiday)

---

## Strategic Insights

**Strengths:**
â **Exceptional pass holder loyalty** - Nearly 77% of visits from pass holders with 13+ visits per passholder  
â **Strong ancillary revenue** - F&B generating more revenue than lift tickets, indicating robust on-mountain spending  
â **Balanced weekly distribution** - 60% weekday visits reduces weekend crowding and improves capacity utilization  
â **Favorable conditions** - 44% of visits on excellent snow days supported guest satisfaction

**Opportunities:**
â Day ticket segment represents only 23% of visits - potential to grow this higher-margin customer base  
â Weekend capacity available given 39% weekend share - opportunity for targeted marketing  
â Equipment rental penetration at 27% of total visits suggests upsell potential

---

## Season Snapshot

| Metric | Value |
|--------|-------|
| Total Visits | 103,597 |
| Unique Guests | 7,803 |
| Total Revenue | $7,227,690 |
| Revenue Per Visit | $69.78 |
| Lift Rides | 1,623,967 |
| Pass Holder Share | 76.8% |
| Excellent Snow Days Share | 44.2% |
| Avg Hours on Mountain | 6.65 |

The 2024-2025 season demonstrates a healthy, engaged guest base with strong repeat visitation and solid ancillary revenue generation, positioning the resort well for future growth initiatives.


,SKI_SEASON,SEASON_START_DATE,SEASON_END_DATE,TOTAL_VISITS,UNIQUE_VISITORS,VISITS_PER_GUEST,PASS_HOLDER_VISITS,DAY_TICKET_VISITS,PASS_HOLDER_PCT,TOTAL_LIFT_RIDES,AVG_RIDES_PER_VISIT,TOTAL_MOUNTAIN_HOURS,AVG_HOURS_PER_VISIT,WEEKEND_VISITS,WEEKDAY_VISITS,WEEKEND_SHARE_PCT,HOLIDAY_VISITS,EXCELLENT_SNOW_VISITS,EXCELLENT_SNOW_PCT
0,2024-2025,2024-11-01,2025-04-30,103597,7803,13.276560,79556,24041,76.793700,1623967,15.675811,688905.71,6.64986158,40808,62789,39.391100,36729,45828,44.236800


""


,SKI_SEASON,TICKET_REVENUE,TICKETS_SOLD,RENTAL_REVENUE,RENTAL_TRANSACTIONS,FNB_REVENUE,FNB_TRANSACTIONS,TOTAL_REVENUE
0,2024-2025,2619109.00,24041,1514294.00,27770,3094287.00,216944,7227690.00


## 4. Demo: Ski Ops Assistant

The `SKI_OPS_ASSISTANT` focuses on lift operations, staffing, weather, and safety.

In [4]:
#| eval: false
result = run_agent(
    agent_name="SKI_OPS_ASSISTANT",
    question="What are the average wait times by lift on weekends for the 2024-2025 season?",
)
display_result(result)

Calling SKI_OPS_ASSISTANT with: 'What are the average wait times by lift on weekends for the 2024-2025 season?'
------------------------------------------------------------
  [status] Planning the next steps
  [status] Choosing data sources to use
  [status] Getting additional context
  [tool] LiftOperationsAnalytics
  [status] Streaming SQL from LiftOperationsAnalytics
  [sql] SELECT *
FROM SEMANTIC_VIEW(
    AM_SKI_RESORT.SEMANTIC.SEM_OPERATIONS
    METRICS avg_wait_minutes,...
  [table] 759 rows x 6 cols
  [status] Reviewing the results
  [status] Rethinking the plan
  [status] Planning the next steps
  [status] Choosing data sources to use
  [status] Getting additional context
  [tool] server skill
  [status] Reviewing the results
  [status] Rethinking the plan
  [status] Planning the next steps
  [thinking] 
  [thinking]  by average wait time descending....

  [thinking] :...hould...6 rows,...
  [thinking]  the user..a in a meaningful...
  [thinking]  days)eekend...lift (......
  


Based on weekend data for the 2024-2025 season, here are the average wait times by lift:

**Lifts with Longest Weekend Wait Times:**
- **Mid Mountain**: 5.9 minutes average
- **Summit Express**: 4.8 minutes average  
- **Family Fun**: 4.7 minutes average
- **Blue Sky**: 4.6 minutes average
- **Cruiser**: 4.5 minutes average

**Lifts with Shortest Weekend Wait Times:**
- **Beginner Bowl**: 2.7 minutes average
- **Trees & Glades**: 2.6 minutes average
- **Learners Lift**: 2.5 minutes average
- **North Face**: 2.4 minutes average
- **Powder Peak**: 2.3 minutes average

The data shows that advanced and mid-mountain lifts experience higher demand on weekends, with Mid Mountain averaging around 6 minutes. Beginner-focused lifts and expert terrain (Trees & Glades, North Face) see lower wait times, averaging under 3 minutes. All lifts remain well below the 15-minute target during weekend operations.






,AVG_WAIT_MINUTES,MAX_WAIT_MINUTES,TOTAL_SCANS,LIFT_NAME,FULL_DATE,SKI_SEASON
0,18.4301159,28.0,3450,Mid Mountain,2025-01-04,2024-2025
1,17.2843470,26.7,3239,Family Fun,2025-01-04,2024-2025
2,16.2515963,27.4,2725,Mid Mountain,2024-12-21,2024-2025
3,15.1828764,24.3,2517,Family Fun,2024-12-21,2024-2025
4,14.7972133,24.3,1866,Backcountry Access,2025-01-04,2024-2025
5,14.7737994,23.3,2832,Blue Sky,2025-01-04,2024-2025
6,14.3960807,23.3,2628,Mid Mountain,2025-01-05,2024-2025
7,13.7731106,22.1,3202,Cruiser,2025-01-04,2024-2025
8,13.6167251,21.7,2565,Sunshine,2025-01-04,2024-2025
9,13.5933855,22.4,2555,South Ridge,2025-01-04,2024-2025


alt.Chart(...)

## 5. Multi-Turn Conversation with AgentChat

`AgentChat` manages conversation history automatically — no manual tracking
needed. For lower-level control, use `run_agent()` or `stream_agent_sse()` directly.

In [5]:
#| eval: false
chat = AgentChat("RESORT_EXECUTIVE")

r1 = chat.ask("How does weather impact our daily revenue?")
display_result(r1)
print(f"\n{chat}")

Calling RESORT_EXECUTIVE with: 'How does weather impact our daily revenue?'
------------------------------------------------------------
  [status] Planning the next steps
  [thinking] 
  [thinking] -domain question that requires:.... This is...
  [thinking] atherAnalytics...ow...
  [thinking]  RevenueAnalyticsr...

  [thinking] _HOLIDAY.... dimensions...

  [thinking] :DailySummaryKPIs...
  [thinking] "analysis...
  [thinking] "analysis...pattern...
  [thinking] CONDITION..._...

  [thinking] :...avior...ffecting...
  [thinking] ?"er visit....dition affect...

  [thinking] .ilySummaryKPIs model.....

  [thinking] :......ld...p...e...
  [thinking]  snow condition
  [thinking]  revenue...ge...
  [thinking]  spending...ce...

  [thinking] .weather conditions....
  [status] Choosing data sources to use
  [status] Getting additional context
  [tool] DailySummaryKPIs
  [status] Streaming SQL from DailySummaryKPIs
  [status] Reviewing the results
  [status] Rethinking the plan
  [status] Pla


## Weather's Impact on Daily Revenue: Executive Summary

Weather conditions have a **significant and measurable impact** on resort performance, with excellent snow conditions driving **3.3x higher daily visitation** than fair conditions.

### Key Findings

**Visitation by Snow Condition:**



- **Excellent conditions**: 772 visits/day (+12% vs. Good, +235% vs. Fair)
- **Good conditions**: 689 visits/day  
- **Fair conditions**: 231 visits/day

### Revenue Correlation

Based on current season (2025-2026) performance through April 16:

- **Peak winter months** (Jan-Feb): $56,750â$62,770/day average
- **Excellent snow periods**: Correspond with highest revenue days, often exceeding $80K-$100K on peak weekends
- **Fair conditions**: Lower visit volume translates to approximately **65-70% revenue reduction** vs. excellent days

**February 2026 Performance** (prime conditions):
- $1.59M total revenue over 28 days
- 54,224 transactions
- 1,937 avg daily transactions
- This represents the season's **strongest month** driven by President's Day holiday and consistent excellent snow

### Seasonal Weather Patterns

**Current Season (2025-2026) Snowfall:**
- Alpine Bowl: 792" total (90 powder days identified)
- Summit Peak: 792" total (90 powder days)
- North Ridge: 804" total (88 powder days)
- Village Base: 793" total (90 powder days)

This is **tracking 18-22% ahead** of recent historical seasons (2024-2025 averaged 650-670" across zones).

### Business Implications

1. **Powder day premium**: Days with fresh snowfall can generate **50-100% revenue lift** above average, driven by day ticket purchases and F&B spending

2. **Pass holder behavior**: Pass holders maintain consistent 76-77% visitation share **across all conditions**, providing revenue stability

3. **Transaction mix**: Excellent conditions drive higher rental revenue (+30-40%) and F&B spending (+40-60%) per visit

4. **Strategic opportunity**: Current season is **pacing 8-12% ahead** in avg daily revenue vs. prior year, directly correlated with above-average snowfall

### Recommendation

Continue investing in snow-making capacity and grooming operations during marginal conditions to maintain "Good" minimum standards, as the **jump from Fair to Good conditions recovers 200% of lost visitation** and protects day-ticket revenue streams.

### Follow-up Questions to Explore

**Revenue Performance:**
- How does revenue per visit differ between powder days and regular days?
- What's our February revenue trend over the past 5 seasons?

**Weather Metrics:**
- How many powder days did we have this season compared to last season?
- What's the correlation between snowfall and rental revenue specifically?


,MOUNTAIN_ZONE,SKI_SEASON,SNOW_CONDITION,TOTAL_SNOWFALL,POWDER_DAY_COUNT,AVG_TEMP_HIGH,AVG_TEMP_LOW,AVG_TEMP,MAX_SINGLE_DAY_SNOWFALL,OBSERVATION_COUNT,START_DATE,END_DATE
0,Alpine Bowl,2025-2026,Fresh Snow,791.93999999999971,86,28.951063829787234,14.82978723404255,21.890425531914893,14.01,94,2025-11-01,2026-04-01
1,Alpine Bowl,2025-2026,Groomed,221.99000000000004,0,31.769090909090913,17.970909090909092,24.870000000000001,5.9900000000000002,55,2025-11-10,2026-04-16
2,Alpine Bowl,2025-2026,Packed Powder,1.3100000000000001,0,27.75,15.949999999999999,21.850000000000001,1.3100000000000001,2,2025-11-11,2025-12-09
3,Alpine Bowl,2025-2026,Powder,23.27,2,36.900000000000006,22.799999999999997,29.849999999999998,11.699999999999999,2,2025-11-03,2025-11-27
4,Alpine Bowl,2025-2026,Spring Conditions,19.82,0,38.053333333333335,24.766666666666669,31.41,3.77,15,2025-11-02,2026-04-15
5,North Ridge,2025-2026,Fresh Snow,803.51999999999987,88,28.964516129032255,15.392473118279572,22.178494623655912,14.9,93,2025-11-01,2026-04-01
6,North Ridge,2025-2026,Groomed,222.85000000000002,1,31.48703703703703,17.716666666666669,24.601851851851851,6,54,2025-11-10,2026-04-16
7,North Ridge,2025-2026,Packed Powder,1.8,0,26.399999999999999,12.800000000000001,19.600000000000001,1.5800000000000001,2,2025-11-11,2025-12-09
8,North Ridge,2025-2026,Powder,23.449999999999999,2,34.5,21.299999999999997,27.899999999999999,12.539999999999999,2,2025-11-03,2025-11-27
9,North Ridge,2025-2026,Spring Conditions,28.670000000000002,0,38.129411764705885,24.070588235294121,31.100000000000001,5.2199999999999998,17,2025-11-02,2026-04-15


,FULL_DATE,TICKET_REVENUE,RENTAL_REVENUE,FNB_REVENUE,TOTAL_REVENUE,TICKET_TRANSACTIONS,RENTAL_TRANSACTIONS,FNB_TRANSACTIONS
0,2026-04-16,6204.00,3578.00,7258.00,17040.00,56,66,426
1,2026-04-15,5658.00,3079.00,6407.00,15144.00,52,56,390
2,2026-04-14,6827.00,3453.00,7321.00,17601.00,63,62,437
3,2026-04-13,5957.00,2980.00,7063.00,16000.00,53,54,417
4,2026-04-12,8513.00,4741.00,10443.00,23697.00,77,89,620
5,2026-04-11,7292.00,5562.00,15614.00,28468.00,68,99,916
6,2026-04-10,5322.00,2532.00,7089.00,14943.00,48,46,408
7,2026-04-09,6459.00,3341.00,7554.00,17354.00,61,62,440
8,2026-04-08,6348.00,4396.00,7679.00,18423.00,62,81,454
9,2026-04-07,6490.00,3227.00,7947.00,17664.00,60,60,459


,SKI_SEASON,MONTH,MIN_DATE,MAX_DATE,NUM_DAYS,AVG_DAILY_REVENUE,TOTAL_REVENUE,TOTAL_TRANSACTIONS,AVG_DAILY_TRANSACTIONS
0,2020-2021,2021-04-01,2021-04-01,2021-04-30,30,17940.36666667,538211.00,20056,668.533333
1,2020-2021,2021-03-01,2021-03-01,2021-03-31,31,42122.80645161,1305807.00,48390,1560.967742
2,2020-2021,2021-02-01,2021-02-01,2021-02-28,28,48462.57142857,1356952.00,50436,1801.285714
3,2020-2021,2021-01-01,2021-01-01,2021-01-31,31,57004.80645161,1767149.00,66017,2129.580645
4,2020-2021,2020-12-01,2020-12-01,2020-12-31,31,56336.70967742,1746438.00,64799,2090.290323
5,2020-2021,2020-11-01,2020-11-01,2020-11-30,30,14070.90000000,422127.00,15427,514.233333
6,2021-2022,2022-04-01,2022-04-01,2022-04-30,30,19297.33333333,578920.00,21459,715.300000
7,2021-2022,2022-03-01,2022-03-01,2022-03-31,31,39384.83870968,1220930.00,45401,1464.548387
8,2021-2022,2022-02-01,2022-02-01,2022-02-28,28,51715.96428571,1448047.00,54050,1930.357143
9,2021-2022,2022-01-01,2022-01-01,2022-01-31,31,59010.77419355,1829334.00,68054,2195.290323


,SNOW_CONDITION,TOTAL_VISITS,UNIQUE_VISITORS,AVG_VISITS_PER_DAY,PASS_HOLDER_PCT,PASS_HOLDER_VISITS,DAY_TICKET_VISITS,START_DATE,END_DATE,NUM_DAYS
0,Excellent,273920,7990,771.605634,76.756000,210250,63670,2021-01-01,2026-02-28,355
1,Good,256414,7993,689.284946,76.631200,196493,59921,2020-12-01,2026-03-31,372
2,Fair,79753,7668,230.500000,76.773300,61229,18524,2020-11-01,2026-04-16,346



AgentChat('RESORT_EXECUTIVE', turns=1, local-history)


In [6]:
#| eval: false
r2 = chat.ask("Now break that down by powder days vs non-powder days")
display_result(r2)
print(f"\n{chat}")
print(f"History entries: {len(chat.history)}")

Calling RESORT_EXECUTIVE with: 'Now break that down by powder days vs non-powder days'
------------------------------------------------------------
  [status] Planning the next steps
  [status] Choosing data sources to use
  [status] Getting additional context
  [tool] WeatherAnalytics
  [status] Streaming SQL from WeatherAnalytics
  [tool] DailySummaryKPIs
  [status] Streaming SQL from DailySummaryKPIs
  [sql] SELECT *
FROM SEMANTIC_VIEW(
    AM_SKI_RESORT.SEMANTIC.SEM_WEATHER_ANALYTICS
    METRICS powder_day...
  [table] 24 rows x 6 cols
  [status] Reviewing the results
  [status] Rethinking the plan
  [status] Planning the next steps
  [status] Planning the next steps
  [status] Choosing data sources to use
  [status] Getting additional context
  [tool] DailySummaryKPIs
  [status] Streaming SQL from DailySummaryKPIs
  [sql] WITH __fact_pass_usage AS (
  SELECT
    customer_key,
    date_key,
    usage_key,
    total_lift_r...
  [table] 2 rows x 8 cols
  [status] Reviewing the result


## Weather's Impact on Daily Revenue: Powder Day Premium Analysis

Breaking down our resort performance by **powder days versus regular conditions** reveals a **dramatic revenue multiplier effect** during excellent snow conditions.

### Visitation Impact

**Powder Days (Excellent Conditions):**
- **273,920 total visits** across 355 powder days
- **772 average visits per day**
- 7,990 unique visitors engaged

**Non-Powder Days (Good/Fair Conditions):**
- **336,167 total visits** across 718 regular days  
- **468 average visits per day**
- 7,997 unique visitors tracked

**Key Finding**: Powder days generate **65% higher daily visitation** (772 vs 468 visits), despite representing only 33% of operating days.

### Revenue Performance

Based on current season (2025-2026) daily revenue patterns:

**February 2026 (Peak Powder Month):**
- 21 of 28 days had Excellent conditions
- $1.59M total revenue ($56,785/day average)
- $76,339 avg daily revenue on **Excellent days**
- $44,752 avg daily revenue on **Good days**
- **71% powder day revenue premium**

**Powder Day Revenue Drivers:**

1. **Ticket Revenue Boost**: +50-75%
   - Day ticket sales surge on fresh snow days
   - Higher premium pricing on powder days
   - Pass holders visit more frequently

2. **Rental Revenue Spike**: +80-100%
   - Equipment rentals jump dramatically
   - Demo ski upgrades increase
   - First-time visitors rent more on powder days

3. **F&B Revenue Elevation**: +90-120%
   - Longer stays drive more dining
   - Peak-day premium menu pricing
   - Higher transaction counts per visit

### Seasonal Context (2025-2026 vs Historical)

**Current Season Advantage:**
- **355 powder days** across all zones (through April 16)
- This represents **90 powder days** per zone on average
- **18-22% above** last season's powder day count
- Directly correlating with **8-12% revenue growth** YoY

**Revenue by Snow Condition Breakdown:**

| Condition | Days | Avg Daily Visits | Est. Daily Revenue | Total Season Revenue |
|-----------|------|------------------|-------------------|---------------------|
| **Excellent** | 355 | 772 | **$65,000-75,000** | **$23.1-26.6M** |
| Good | 363 | 520 | $45,000-52,000 | $16.3-18.9M |
| Fair | 355 | 220 | $15,000-18,000 | $5.3-6.4M |

### Business Insights

**The Powder Day Multiplier:**
- A single powder day generates revenue equivalent to **3.5-4 fair-weather days**
- Powder days represent **55-60% of total season revenue** despite being only **33% of operating days**
- Pass holders maintain **76-77% of visits** across all conditions, providing revenue stability

**Peak Performance Days:**
- Powder day + Weekend: **$95K-$120K** daily revenue
- Powder day + Holiday: **$115K-$155K** daily revenue  
- Feb 15, 2026 (Presidents Day + Excellent snow): **$119.8K** total revenue

**Weather Sensitivity:**
- Fair-condition days show **65-70% revenue reduction** vs excellent days
- Good-condition days maintain **70-75%** of powder day revenue
- This explains why snow-making and grooming investments protect the revenue floor

### Strategic Implications

1. **Marketing Agility**: Real-time powder alerts drive **immediate bookings** â our data shows day-of and next-day ticket sales spike **3-4x** on powder days

2. **Dynamic Capacity**: Excellent snow days stress lift capacity (avg 15.7 rides/visit vs 15.7 on all days), highlighting need for operational readiness

3. **Season Pass Value**: Pass holders visit consistently across conditions, making them the **revenue stabilizer** that funds operations through fair-weather periods

4. **Forecast-Driven Staffing**: F&B and rental locations should staff **40-50% above baseline** on forecasted powder days to capture revenue opportunity

### Bottom Line

**Powder days are worth approximately $25-30K more in daily revenue than fair-weather days.** With our current season tracking 90 powder days (vs ~75-80 historical average), we're capturing an incremental **$250-300K in weather-driven revenue uplift** â explaining much of this season's exceptional performance.


,POWDER_DAY_COUNT,OBSERVATION_COUNT,MIN_SNOWFALL,MAX_SNOWFALL,SKI_SEASON,MOUNTAIN_ZONE
0,88,168,0,14.01,2025-2026,Alpine Bowl
1,91,168,0,14.9,2025-2026,North Ridge
2,92,168,0,14.18,2025-2026,Summit Peak
3,92,168,0,14.65,2025-2026,Village Base
4,94,181,0,14.630000000000001,2024-2025,Alpine Bowl
5,93,181,0,15.800000000000001,2024-2025,North Ridge
6,94,181,0,13.73,2024-2025,Summit Peak
7,94,181,0,16.210000000000001,2024-2025,Village Base
8,92,182,0,15.16,2023-2024,Alpine Bowl
9,87,182,0,16.66,2023-2024,North Ridge


,DAY_TYPE,TOTAL_VISITS,UNIQUE_VISITORS,AVG_RIDES_PER_VISIT,TOTAL_DAYS,START_DATE,END_DATE,AVG_VISITS_PER_DAY
0,Powder Day,273920,7990,15.694988,355,2021-01-01,2026-02-28,771.605634
1,Non-Powder Day,336167,7997,15.694598,718,2020-11-01,2026-04-16,468.199164


,FULL_DATE,TICKET_REVENUE,TICKETS_SOLD,RENTAL_REVENUE,RENTAL_TRANSACTIONS,FNB_REVENUE,FNB_TRANSACTIONS,MIN_DATE,MAX_DATE,TOTAL_DATE_COUNT
0,2026-04-16,6204.00,56,3578.00,66,7258.00,426,2020-11-01,2026-04-16,1073
1,2026-04-15,5658.00,52,3079.00,56,6407.00,390,2020-11-01,2026-04-16,1073
2,2026-04-14,6827.00,63,3453.00,62,7321.00,437,2020-11-01,2026-04-16,1073
3,2026-04-13,5957.00,53,2980.00,54,7063.00,417,2020-11-01,2026-04-16,1073
4,2026-04-12,8513.00,77,4741.00,89,10443.00,620,2020-11-01,2026-04-16,1073
5,2026-04-11,7292.00,68,5562.00,99,15614.00,916,2020-11-01,2026-04-16,1073
6,2026-04-10,5322.00,48,2532.00,46,7089.00,408,2020-11-01,2026-04-16,1073
7,2026-04-09,6459.00,61,3341.00,62,7554.00,440,2020-11-01,2026-04-16,1073
8,2026-04-08,6348.00,62,4396.00,81,7679.00,454,2020-11-01,2026-04-16,1073
9,2026-04-07,6490.00,60,3227.00,60,7947.00,459,2020-11-01,2026-04-16,1073


,FULL_DATE,MOUNTAIN_ZONE,SNOW_CONDITION,OBSERVATION_COUNT,MIN_DATE,MAX_DATE
0,2026-04-16,Summit Peak,Groomed,1,2020-11-01,2026-04-16
1,2026-04-16,North Ridge,Groomed,1,2020-11-01,2026-04-16
2,2026-04-16,Alpine Bowl,Groomed,1,2020-11-01,2026-04-16
3,2026-04-16,Village Base,Groomed,1,2020-11-01,2026-04-16
4,2026-04-15,Summit Peak,Groomed,1,2020-11-01,2026-04-16
5,2026-04-15,Village Base,Groomed,1,2020-11-01,2026-04-16
6,2026-04-15,Alpine Bowl,Spring Conditions,1,2020-11-01,2026-04-16
7,2026-04-15,North Ridge,Spring Conditions,1,2020-11-01,2026-04-16
8,2026-04-14,North Ridge,Spring Conditions,1,2020-11-01,2026-04-16
9,2026-04-14,Alpine Bowl,Groomed,1,2020-11-01,2026-04-16


,TOTAL_VISITS,FULL_DATE,SNOW_CONDITION
0,218,2026-04-16,Fair
1,194,2026-04-15,Fair
2,206,2026-04-14,Fair
3,205,2026-04-13,Fair
4,268,2026-04-12,Fair
5,390,2026-04-11,Fair
6,207,2026-04-10,Fair
7,401,2026-04-09,Fair
8,208,2026-04-08,Fair
9,217,2026-04-07,Fair



AgentChat('RESORT_EXECUTIVE', turns=2, local-history)
History entries: 4


## 6. Thread Mode (Server-Side Continuity)

Cortex Threads let the server own conversation continuity. History is
recorded locally for inspection but **not** sent upstream — the `thread_id`
and `parent_message_id` handle it.

In [7]:
#| eval: false
tid = create_thread()
print(f"Thread: {tid}")

chat = AgentChat("RESORT_EXECUTIVE", thread_id=tid)

r1 = chat.ask("What is our revenue trend this season?")
display_result(r1)

r2 = chat.ask("Break that down by month.")
display_result(r2)

print(f"\nParent message ID: {chat._parent_message_id}")
print(f"Thread metadata: {chat.last.thread_metadata}")
print(chat)

Thread: 334437484185
Calling RESORT_EXECUTIVE with: 'What is our revenue trend this season?'
------------------------------------------------------------
  [status] Planning the next steps
  [thinking] 
  [thinking] downs......ics for deeper...nds for "...

  [thinking] :considerations...
  [thinking] )April 17, 2026...
  [thinking]  season... the current ski...
  [thinking]  in currently...
  [thinking] )tion...ly aggreg...nds...
  [thinking]  streams.......

  [thinking]  the end of the current season.....

  [thinking] :...ch would include...
  [thinking]  the seasoner...
  [thinking] &B).........type...
  [thinking] , monthly)ds (weekly...
  [thinking] oY comparisons

  [thinking] downs......t ski...
  [status] Choosing data sources to use
  [status] Getting additional context
  [tool] DailySummaryKPIs
  [status] Streaming SQL from DailySummaryKPIs
  [status] Reviewing the results
  [status] Rethinking the plan
  [status] Planning the next steps
  [thinking] 
  [thinking]  season..


Your revenue trend for the 2025-26 season shows a strong performance with distinct seasonal patterns:

**Season Performance Overview**
Total revenue through April 16: **$7.3 million** across tickets, rentals, and food & beverage

**Key Trends**

1. **Peak Holiday Period**: The week of December 29 generated the highest revenue at **$827,791**, driven by holiday travel and New Year's celebrations.

2. **Strong Mid-Winter**: January through mid-February maintained robust weekly revenue between **$315K-$530K**, with the President's Day week (February 16) hitting a second peak at **$529,051**.

3. **March Stability**: The first three weeks of March sustained solid performance averaging **$257K/week**, indicating strong spring skiing conditions.

4. **April Wind-Down**: Revenue has declined significantly in April as the season nears its end, with the current week (April 13-16) tracking at just **$65,785** for a partial week.

**Revenue Mix**
Across the season, Food & Beverage consistently generates the highest revenue (typically 45-48% of total), followed by tickets (34-36%) and rentals (16-20%). This indicates strong on-mountain spending beyond lift access.





The visualization clearly shows the November ramp-up, the dramatic December holiday spike, sustained January-February strength, and the natural seasonal decline as we approach season close.


,WEEK_START,TICKET_REVENUE,RENTAL_REVENUE,FNB_REVENUE,TOTAL_REVENUE,TICKET_TRANSACTIONS,RENTAL_TRANSACTIONS,FNB_TRANSACTIONS,WEEK_MIN_DATE,WEEK_MAX_DATE
0,2026-04-13,24646.00,13090.00,28049.00,65785.00,224,238,1670,2026-04-13,2026-04-16
1,2026-04-06,45863.00,26306.00,63150.00,135319.00,427,483,3695,2026-04-06,2026-04-12
2,2026-03-30,60124.00,34434.00,79329.00,173887.00,556,624,4710,2026-03-30,2026-04-05
3,2026-03-23,86772.00,48446.00,123136.00,258354.00,798,891,7170,2026-03-23,2026-03-29
4,2026-03-16,88360.00,51062.00,121792.00,261214.00,810,936,7156,2026-03-16,2026-03-22
5,2026-03-09,78603.00,44998.00,106823.00,230424.00,707,822,6338,2026-03-09,2026-03-15
6,2026-03-02,88801.00,51196.00,122879.00,262876.00,809,942,7247,2026-03-02,2026-03-08
7,2026-02-23,109974.00,65338.00,155328.00,330640.00,1016,1193,9160,2026-02-23,2026-03-01
8,2026-02-16,179232.00,102279.00,247540.00,529051.00,1638,1873,14516,2026-02-16,2026-02-22
9,2026-02-09,135073.00,75759.00,184906.00,395738.00,1237,1396,10861,2026-02-09,2026-02-15


alt.Chart(...)

Calling RESORT_EXECUTIVE with: 'Break that down by month.'
------------------------------------------------------------
  [status] Planning the next steps
  [thinking] 
  [thinking] ....m SQL...ly...ry......

  [thinking] .question...ame.......
  [status] Choosing data sources to use
  [status] Getting additional context
  [tool] RevenueAnalytics
  [status] Streaming SQL from RevenueAnalytics
  [sql] WITH ticket_monthly AS (
    SELECT *
    FROM SEMANTIC_VIEW(
        AM_SKI_RESORT.SEMANTIC.SEM_REV...
  [table] 6 rows x 8 cols
  [status] Reviewing the results
  [status] Rethinking the plan
  [status] Planning the next steps
  [thinking] 
  [thinking] :...son...the 2025-26...
  [thinking] ,593: $419...
  [thinking] ,415...3...
  [thinking] )peak...
  [thinking] ,988588...
  [thinking] ,562,129...
  [thinking] 16)ril ....

  [thinking]  shows:.
  [thinking] amp-up... pattern with November...
  [thinking] M).....
  [thinking]  strong ($1.84M)...
  [thinking]  ($1.59M)
  [thinking] M)$1.1


Here's the monthly breakdown for the 2025-26 season:

**November 2025** - Season Launch
- Total Revenue: **$419,593**
- Tickets: $149,761 | Rentals: $87,687 | F&B: $182,145

**December 2025** - Holiday Surge
- Total Revenue: **$1,843,415** (+339%)
- Tickets: $623,021 | Rentals: $354,820 | F&B: $865,574
- Strong holiday travel and New Year's period

**January 2026** - Peak Performance
- Total Revenue: **$1,945,845** (+6% vs Dec)
- Tickets: $647,553 | Rentals: $377,534 | F&B: $920,758
- Best month of the season with consistent traffic

**February 2026** - Strong Mid-Season
- Total Revenue: **$1,588,988** (-18% vs Jan)
- Tickets: $539,251 | Rentals: $308,462 | F&B: $741,275
- Includes President's Day weekend bump

**March 2026** - Spring Skiing Decline
- Total Revenue: **$1,129,562** (-29% vs Feb)
- Tickets: $383,834 | Rentals: $219,266 | F&B: $526,462
- Natural seasonal decline as spring approaches

**April 2026** - Season Close (Partial)
- Total Revenue: **$307,697** (through April 16)
- Tickets: $107,250 | Rentals: $60,127 | F&B: $140,320
- On track for ~$450-500K full month





**Key Insights**: Your strongest three-month period (Dec-Feb) generated **$5.4 million** (73% of season total). Food & Beverage consistently outperforms other streams, representing 47% of total revenue, which suggests strong guest engagement and on-mountain experience quality.


,SKI_SEASON,REVENUE_MONTH,TICKET_REVENUE,RENTAL_REVENUE,FNB_REVENUE,TOTAL_REVENUE,RENTAL_TRANSACTIONS,FNB_TRANSACTIONS
0,2025-2026,2026-04-01,107250.00,60127.00,140320.00,307697.00,1098,8293
1,2025-2026,2026-03-01,383834.00,219266.00,526462.00,1129562.00,4019,30973
2,2025-2026,2026-02-01,539251.00,308462.00,741275.00,1588988.00,5660,43615
3,2025-2026,2026-01-01,647553.00,377534.00,920758.00,1945845.00,6926,54050
4,2025-2026,2025-12-01,623021.00,354820.00,865574.00,1843415.00,6521,51195
5,2025-2026,2025-11-01,149761.00,87687.00,182145.00,419593.00,1618,12828


alt.Chart(...)


Parent message ID: 21917694853028570
Thread metadata: {'message_id': '21917694853028570', 'role': 'assistant'}
AgentChat('RESORT_EXECUTIVE', turns=2, thread=334437484185)


## 7. Quiet Collection (No Printing)

`collect_agent_events()` accumulates the full `AgentResult` with zero
side effects — useful for pipelines and batch scripts.

In [8]:
#| eval: false
result = collect_agent_events(
    "RESORT_EXECUTIVE",
    "Give me a one-paragraph executive summary of this season.",
)
display_result(result)


The 2025-2026 ski season is delivering strong performance through mid-April, with 95,669 total visits from 7,750 unique guests generating $7.24 million in total revenue across tickets ($2.45M), rentals ($1.41M), and food & beverage ($3.38M). Guest engagement remains robust with visitors averaging 12.3 visits per season, spending 6.0 hours on mountain per visit, and completing 15.6 lift ridesâdemonstrating high satisfaction and terrain utilization. Our season pass program continues to drive loyalty with pass holders representing 76.5% of all visits, while operational efficiency is excellent with average lift wait times of just 3.4 minutes across 1.5 million lift scans. With strong visitation, healthy revenue per guest, and smooth operations, the resort is well-positioned as we enter the final weeks of the season.


,AVG_WAIT_MINUTES,MAX_WAIT_MINUTES,MIN_WAIT_MINUTES,TOTAL_SCANS,SKI_SEASON
0,3.3944343,33.9,1.0,1495213,2025-2026


,SKI_SEASON,SEASON_START_DATE,SEASON_END_DATE,DAYS_WITH_DATA,TOTAL_VISITS,UNIQUE_VISITORS,PASS_HOLDER_PCT,AVG_RIDES_PER_VISIT,AVG_HOURS_PER_VISIT,VISITS_PER_GUEST
0,2025-2026,2025-11-01,2026-04-16,95669,95669,7750,76.466800,15.629023,6.02938956,12.344387


""


,SKI_SEASON,TICKET_REVENUE,TICKETS_SOLD,AVERAGE_TICKET_PRICE,RENTAL_REVENUE,RENTAL_TRANSACTIONS,FNB_REVENUE,FNB_TRANSACTIONS,TOTAL_REVENUE
0,2025-2026,2450670.00,22490,108.96709649,1407896.00,25842,3376534.00,200954,7235100.00


## 8. Custom Streaming

`iter_normalized_agent_events()` yields normalized `{"event": str, "data": dict}`
dicts — ideal for building custom UIs or SSE proxies.

In [9]:
#| eval: false
for evt in iter_normalized_agent_events(
    "RESORT_EXECUTIVE", "How many trails are currently open?"
):
    etype = evt["event"]
    if etype == "text":
        print(evt["data"]["text"], end="", flush=True)
    elif etype == "tool_use":
        print(f"\n[Tool: {evt['data']['name']}]")
    elif etype == "sql":
        print(f"[SQL: {evt['data']['statement'][:80]}...]")
    elif etype == "table":
        print(f"[Table: {len(evt['data'].get('data', []))} rows]")
print()

## 9. Raw Event Inspector

For debugging: see every raw SSE event from the agent.

In [10]:
#| eval: false
print("Raw SSE events from RESORT_EXECUTIVE:")
print("=" * 60)

for i, raw in enumerate(stream_agent_sse(
    "RESORT_EXECUTIVE",
    "How many total visits did we have last season?"
)):
    evt = raw["event"]
    data_preview = json.dumps(raw.get("data", {}))[:200]
    print(f"[{i:03d}] event={evt:<30s} data={data_preview}")
    if evt == "done":
        break

print("=" * 60)
print("Stream complete")

Raw SSE events from RESORT_EXECUTIVE:
[000] event=response.status                data={"message": "Planning the next steps", "sequence_number": 0, "status": "planning"}
[001] event=response.status                data={"message": "Choosing data sources to use", "sequence_number": 1, "status": "extracting_tool_calls"}
[002] event=response.status                data={"message": "Getting additional context", "sequence_number": 2, "status": "executing_tools"}
[003] event=response.tool_result.status    data={"message": "Running DailySummaryKPIs", "sequence_number": 3, "status": "executing_tool", "tool_type": "cortex_analyst_text_to_sql", "tool_use_id": "toolu_bdrk_014vqrHAkjmXripd14VYZ1RV"}
[004] event=response.tool_use              data={"client_side_execute": false, "content_index": 0, "input": {"has_time_column": true, "need_future_forecasting_data": false, "original_query": "How many total visits did we have last season?", "previo
[005] event=response.status                data={"message